# Covertype - Model Pipeline

Perform data preparation step and model training over the dataset `Covertype`.

# Setup Notebook

## Imports

In [1]:
# Import Standard Libraries
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
from dynaconf import Dynaconf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

## Define Configurations

In [2]:
# Read .env file
load_dotenv()

# Setup root path
root_path = Path(os.getenv("DRUIDIC_GROVE_AI_ROOT_PATH"))

print(f"🛤️ Root path: {root_path}")

# Read configurations
config = Dynaconf(
    settings_files=[root_path / "configurations" / "covertype" / "model_pipeline_config.toml"]
)

# Read configurations - Data Sources
covertype_data_path = config["data_sources"]["covertype_data_path"]
covertype_columns = config["data_sources"]["covertype_columns"]
features = config["data_sources"]["features"]
labels = config["data_sources"]["labels"]

# Read configurations - Transformations
features_to_standardise = config["transformations"]["features_to_standardise"]

🛤️ Root path: /Users/simone.porreca/Projects/DruidicGroveAI


## Read Data

In [3]:
# Read data files
covertype_data = pd.read_csv(root_path / covertype_data_path)
covertype_data.columns = covertype_columns

# Data Preparation

## Feature Engineering

### Aspect Trigonometry Transformation

In [4]:
covertype_data['Sin_Aspect'] = np.sin(np.radians(covertype_data['Aspect']))
covertype_data['Cos_Aspect'] = np.cos(np.radians(covertype_data['Aspect']))

### Elevation Discretisation

In [5]:
# Split into 4 quartiles based on your distribution (0-25%, 25-50%, etc.)
covertype_data['Elevation_Zones'] = pd.qcut(
    covertype_data['Elevation'],
    q=4,
    labels=['1', '2', '3', '4'])

# One-Hot Encode
covertype_data = pd.get_dummies(covertype_data, columns=['Elevation_Zones'])

## Train & Test Split

In [6]:
# Define X and y for the training set
X = covertype_data[features]
y = np.ravel(covertype_data[labels])

# Split training data into train and validation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=108)

## Standardisation

In [7]:
# Standardise features
scaler = StandardScaler()
scaler.fit(X_train[features_to_standardise])
X_train[features_to_standardise] = scaler.transform(X_train[features_to_standardise])
X_test[features_to_standardise] = scaler.transform(X_test[features_to_standardise])

## Label Encoding

In [8]:
# Encode the labels
labels_encoder = LabelEncoder()
labels_encoder.fit(y_train)
y_train = labels_encoder.transform(y_train)
y_test = labels_encoder.transform(y_test)